# COSMIC Cancer Mutation Census benchmark construction

**Purpose.** Build the somatic-variant benchmark used for Supplementary Figure S5 from COSMIC Cancer Mutation Census (CMC) v103 and MANE Select v1.5.

## Reproducibility contract

- **Run from:** the repository root.
- **Licensed input:** download `CancerMutationCensus_AllData_v103_GRCh37.tsv.gz` from the [COSMIC Cancer Mutation Census download page](https://cancer.sanger.ac.uk/cosmic/download/cancer-mutation-census) after logging in, and place it at `data/raw/cosmic/`.
- **Public input:** MANE Select GRCh38 v1.5 is downloaded from NCBI with the command below.
- **Output:** `data/processed/cosmic_benchmark.csv` and `results/tables/cosmic_benchmark_summary.csv`.
- **Randomness:** benign variants are sampled with seed 42.
- **Compute:** CPU and substantial memory; model inference is not performed.

COSMIC data are not redistributed in this repository because access is governed by the COSMIC license.

```bash
mkdir -p data/raw/mane
curl -L https://ftp.ncbi.nlm.nih.gov/refseq/MANE/MANE_human/release_1.5/MANE.GRCh38.v1.5.refseq_genomic.gff.gz \
  -o data/raw/mane/MANE.GRCh38.v1.5.refseq_genomic.gff.gz
```


## 1. Setup and validate inputs


In [ ]:
from pathlib import Path
import gc
import gzip
import os
import re
from multiprocessing import Pool

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

REPO_ROOT = Path.cwd()
DATA_DIR = REPO_ROOT / "data"
CMC_GZ = DATA_DIR / "raw" / "cosmic" / "CancerMutationCensus_AllData_v103_GRCh37.tsv.gz"
MANE_GFF = DATA_DIR / "raw" / "mane" / "MANE.GRCh38.v1.5.refseq_genomic.gff.gz"
OUT_PATH = DATA_DIR / "processed" / "cosmic_benchmark.csv"
SUMMARY_PATH = REPO_ROOT / "results" / "tables" / "cosmic_benchmark_summary.csv"

RANDOM_SEED = 42
N_PER_RARE_BIN = 10_000
N_WORKERS = 4

for path in (CMC_GZ, MANE_GFF):
    if not path.exists():
        raise FileNotFoundError(f"Required input not found: {path}. See the download instructions above.")

OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)
print(f"CMC input: {CMC_GZ}")
print(f"MANE input: {MANE_GFF}")


## 2. Load CMC v103 and select SNVs


In [ ]:
cmc = pd.read_csv(CMC_GZ, sep="\t", compression="gzip", low_memory=False)
print(f"Loaded {len(cmc):,} CMC rows and {cmc.shape[1]} columns")

columns = {
    "gene": "GENE_NAME",
    "tier": "MUTATION_SIGNIFICANCE_TIER",
    "mutation_type": "ONTOLOGY_MUTATION_CODE",
    "transcript": "ACCESSION_NUMBER",
    "coordinate": "Mutation genome position GRCh38",
    "ref": "GENOMIC_WT_ALLELE_SEQ",
    "alt": "GENOMIC_MUT_ALLELE_SEQ",
    "gnomad_af": "GNOMAD_GENOMES_AF",
    "hgvsc": "Mutation CDS",
    "hgvsp": "Mutation AA",
}
missing = [column for column in columns.values() if column not in cmc.columns]
if missing:
    raise ValueError(f"CMC v103 input is missing required columns: {missing}")

coordinates = cmc[columns["coordinate"]].astype(str).str.extract(r"^([^:]+):(\d+)-\d+$")
cmc["_CHR"] = coordinates[0]
cmc["_POS"] = coordinates[1]
cmc["_REF"] = cmc[columns["ref"]]
cmc["_ALT"] = cmc[columns["alt"]]

keep = ["_CHR", "_POS", "_REF", "_ALT"] + list(columns.values())
cmc = cmc[list(dict.fromkeys(keep))].copy()

ref = cmc["_REF"].astype(str).str.strip().str.upper()
alt = cmc["_ALT"].astype(str).str.strip().str.upper()
snv_mask = (
    ref.str.len().eq(1) & alt.str.len().eq(1)
    & ref.isin(set("ACGT")) & alt.isin(set("ACGT")) & ref.ne(alt)
)
cmc_snv = cmc.loc[snv_mask].copy()
print(f"Retained {len(cmc_snv):,} SNVs ({100 * len(cmc_snv) / len(cmc):.1f}% of CMC rows)")
del cmc, ref, alt
gc.collect()


## 3. Define pathogenic and stratified benign cohorts


In [ ]:
tier = cmc_snv[columns["tier"]].astype(str).str.strip()

pathogenic = cmc_snv[tier.eq("1")].copy()
pathogenic["label"] = 1
pathogenic["label_source"] = "CMC_Tier1"
pathogenic["af_bin"] = "pathogenic"

benign_pool = cmc_snv[tier.eq("Other")].copy()
benign_pool["_af"] = pd.to_numeric(benign_pool[columns["gnomad_af"]], errors="coerce")
bins = [
    ("no_gnomad", benign_pool["_af"].isna(), N_PER_RARE_BIN),
    ("very_rare", benign_pool["_af"].notna() & (benign_pool["_af"] < 0.001), N_PER_RARE_BIN),
    ("rare", benign_pool["_af"].between(0.001, 0.01, inclusive="left"), N_PER_RARE_BIN),
    ("low_freq", benign_pool["_af"].between(0.01, 0.1, inclusive="left"), N_PER_RARE_BIN),
    ("common", benign_pool["_af"] >= 0.1, None),
]

sampled = []
for name, mask, target in bins:
    pool = benign_pool.loc[mask]
    selected = pool if target is None or len(pool) <= target else pool.sample(target, random_state=RANDOM_SEED)
    selected = selected.copy()
    selected["af_bin"] = name
    sampled.append(selected)
    print(f"{name}: selected {len(selected):,} of {len(pool):,}")

benign = pd.concat(sampled, ignore_index=True).drop(columns="_af")
benign["label"] = 0
benign["label_source"] = "CMC_TierOther_stratified"

df_all = pd.concat([pathogenic, benign], ignore_index=True)
print(f"Combined cohort: {len(df_all):,} variants")


## 4. Normalize coordinates and remove duplicates


In [ ]:
VALID_CHROMS = {str(i) for i in range(1, 23)} | {"X", "Y", "MT", "M"}


def normalize_chromosome(value):
    value = str(value).strip().upper().removeprefix("CHR")
    return f"chr{value}" if value in VALID_CHROMS else f"chr{value.lower()}"


df_all["chrom"] = df_all["_CHR"].map(normalize_chromosome)
df_all["pos"] = pd.to_numeric(df_all["_POS"], errors="coerce").astype("Int64")
df_all["ref"] = df_all["_REF"].astype(str).str.strip().str.upper()
df_all["alt"] = df_all["_ALT"].astype(str).str.strip().str.upper()
df_all["gene"] = df_all[columns["gene"]].astype(str).str.strip()
df_all["hgvsc"] = df_all[columns["hgvsc"]]
df_all["hgvsp"] = df_all[columns["hgvsp"]]
df_all["transcript"] = df_all[columns["transcript"]]
df_all["gnomad_af"] = df_all[columns["gnomad_af"]]
df_all["cmc_tier"] = df_all[columns["tier"]]
df_all["mut_type_raw"] = df_all[columns["mutation_type"]]

df_all = df_all.dropna(subset=["chrom", "pos", "ref", "alt"])
df_all = df_all.drop_duplicates(subset=["chrom", "pos", "ref", "alt", "label"])

key = ["chrom", "pos", "ref", "alt"]
conflicting = df_all.groupby(key)["label"].transform("nunique").gt(1)
df_all = df_all.loc[~(conflicting & df_all["label"].eq(0))].copy()
print(f"Normalized cohort: {len(df_all):,} unique GRCh38 SNVs")


## 5. Annotate variants with MANE Select v1.5


In [ ]:
import os
import re as _re
from multiprocessing import Pool, cpu_count

with gzip.open(MANE_GFF, "rt") as file:
    lines = file.readlines()

columns_gff = ["Chromosome", "Source", "Feature", "Start", "End",
               "Score", "Strand", "Frame", "Attributes"]
data = [line.strip().split("\t") for line in lines if not line.startswith("#")]
MANE = pd.DataFrame(data, columns=columns_gff)

# Parse attributes
fields = ['ID', 'Parent', 'Dbxref', 'Name', 'description', 'gbkey',
          'gene', 'gene_biotype', 'product', 'tag', 'transcript_id']

def parse_attributes(attribute_string):
    attributes = {}
    for field in fields:
        match = _re.search(f'{field}=([^;]+)', attribute_string)
        if match:
            attributes[field] = match.group(1)
    if 'Dbxref' in attributes:
        for value in attributes['Dbxref'].split(','):
            if value.startswith('Ensembl:'):
                attributes['Ensembl'] = value.split(':')[1]
                break
    return attributes

parsed_df = pd.DataFrame(MANE['Attributes'].apply(parse_attributes).tolist())
MANE = pd.concat([MANE, parsed_df], axis=1)

# Promoter
distance_dict = {
    'mRNA': 2000, 'lncRNA': 2000, 'snoRNA': 500,
    'snRNA': 500, 'telomerase_RNA': 1500, 'antisense_RNA': 1500
}
Promoter = MANE[MANE['Feature'].isin(distance_dict.keys())].copy()
Promoter[["Promoter_Start", "Promoter_End"]] = None

def calculate_promoter(row):
    if row["Strand"] == "+":
        tss = int(row["Start"])
        return tss - distance_dict[row['Feature']], tss
    else:
        tss = int(row["End"])
        return tss, tss + distance_dict[row['Feature']]

Promoter[["Promoter_Start", "Promoter_End"]] = Promoter.apply(
    calculate_promoter, axis=1, result_type="expand"
)
Promoter = Promoter.reset_index(drop=True)
print(f"MANE rows: {len(MANE):,}")
print(f"Promoter rows: {len(Promoter):,}")


In [ ]:
annotation_columns = [
    'gene',
    'mRNA', 'mRNA_promoter', 'mRNA_exon',
    'coding_sequence', 'start_codon', 'stop_codon',
    'five_prime_UTR', 'three_prime_UTR', 'mRNA_intron', 'mRNA_splice',
    'lncRNA', 'lncRNA_promoter', 'lncRNA_exon',
    'snRNA', 'snRNA_promoter', 'snRNA_exon',
    'antisenseRNA', 'antisenseRNA_promoter', 'antisenseRNA_exon',
    'telomeraseRNA', 'telomeraseRNA_promoter', 'telomeraseRNA_exon',
    'RNaseMRPRNA', 'RNaseMRPRNA_promoter', 'RNaseMRPRNA_exon',
    'snoRNA', 'snoRNA_promoter', 'snoRNA_exon',
    'other'
]

def match_variants(row):
    chrom = row["Chromosome"]
    pos = int(row["PositionVCF"])
    annotation = MANE[((MANE["Chromosome"] == f'chr{chrom}') &
                       (MANE["Start"].astype(int) <= pos) &
                       (MANE["End"].astype(int) >= pos))]
    annotation_promoter = Promoter[((Promoter["Chromosome"] == f"chr{chrom}") &
                                    (Promoter["Promoter_Start"].astype(int) <= pos) &
                                    (Promoter["Promoter_End"].astype(int) >= pos))]
    if annotation.empty and annotation_promoter.empty:
        row['other'] = 1
        return row
    types = annotation['Feature'].unique()
    types_promoter = annotation_promoter['Feature'].unique()
    if "gene" in types:
        row['gene'] = 1
    if "mRNA" in types:
        row['mRNA'] = 1
        transcript_ids = set(annotation[annotation['Feature'] == "mRNA"]["transcript_id"].dropna())
        row['transcript_set'].update(transcript_ids)
        for transcript_id in transcript_ids:
            strand = annotation[annotation['ID']==f'rna-{transcript_id}']['Strand'].iloc[0]
            annotation_mRNA_exon = annotation[((annotation['Parent']==f'rna-{transcript_id}') &
                                               (annotation['Feature'] == "exon"))]
            annotation_mRNA_CDS = annotation[((annotation['Parent']==f'rna-{transcript_id}') &
                                               (annotation['Feature'] == "CDS"))]
            transcript_exon = MANE[((MANE['Parent']==f'rna-{transcript_id}') &
                                    (MANE['Feature'] == "exon"))]
            transcript_CDS = MANE[((MANE['Parent']==f'rna-{transcript_id}') &
                                   (MANE['Feature'] == "CDS"))]
            if (not annotation_mRNA_CDS.empty) and (not annotation_mRNA_exon.empty):
                row['mRNA_exon'] = 1
                row['coding_sequence'] = 1
                if strand=="+":
                    start_1 = min(transcript_CDS['Start'].astype(int))
                    if (pos <= start_1+2) and (pos >= start_1): row['start_codon'] = 1
                    stop_3 = max(transcript_CDS['End'].astype(int))
                    if (pos >= stop_3-2) and (pos <= stop_3): row['stop_codon'] = 1
                else:
                    start_1 = max(transcript_CDS['End'].astype(int))
                    if (pos >= start_1-2) and (pos <= start_1): row['start_codon'] = 1
                    stop_3 = min(transcript_CDS['Start'].astype(int))
                    if (pos <= stop_3+2) and (pos >= stop_3): row['stop_codon'] = 1
            elif (annotation_mRNA_CDS.empty) and (not annotation_mRNA_exon.empty):
                row['mRNA_exon'] = 1
                if strand=="+":
                    fiveUTR_start = min(transcript_exon['Start'].astype(int))
                    fiveUTR_end = min(transcript_CDS['Start'].astype(int))-1
                    if (pos <= fiveUTR_end) and (pos >= fiveUTR_start): row['five_prime_UTR']=1
                    threeUTR_start = max(transcript_CDS['End'].astype(int))+1
                    threeUTR_end = max(transcript_exon['End'].astype(int))
                    if (pos >= threeUTR_start) and (pos <= threeUTR_end): row['three_prime_UTR']=1
                else:
                    fiveUTR_start = max(transcript_exon['End'].astype(int))
                    fiveUTR_end = max(transcript_CDS['End'].astype(int))+1
                    if (pos >= fiveUTR_end) and (pos <= fiveUTR_start): row['five_prime_UTR']=1
                    threeUTR_start = min(transcript_CDS['Start'].astype(int))-1
                    threeUTR_end = min(transcript_exon['Start'].astype(int))
                    if (pos <= threeUTR_start) and (pos >= threeUTR_end): row['three_prime_UTR']=1
            elif (annotation_mRNA_CDS.empty) and (annotation_mRNA_exon.empty):
                row['mRNA_intron'] = 1
                splice_region = transcript_exon.apply(
                    lambda exon: pos in [int(exon['Start'])-1, int(exon['Start'])-2,
                                         int(exon['End'])+1, int(exon['End'])+2], axis=1)
                if splice_region.any(): row['mRNA_splice'] = 1
    if "mRNA" in types_promoter:
        row['mRNA_promoter'] = 1
        transcript_ids = set(annotation_promoter[annotation_promoter['Feature'] == "mRNA"]["transcript_id"].dropna())
        row['promoter_transcript_set'].update(transcript_ids)
    for rna in ['lncRNA', 'snRNA', 'antisenseRNA', 'telomeraseRNA', 'RNaseMRPRNA', 'snoRNA']:
        if rna in types:
            row[rna]=1
            transcript_ids = set(annotation[annotation['Feature'] == rna]["transcript_id"].dropna())
            row['transcript_set'].update(transcript_ids)
            for transcript_id in transcript_ids:
                annotation_RNA_exon = annotation[((annotation['Parent']==f'rna-{transcript_id}') &
                                                   (annotation['Feature'] == "exon"))]
                if not annotation_RNA_exon.empty: row[f"{rna}_exon"] = 1
        if rna in types_promoter:
            row[f"{rna}_promoter"] = 1
            transcript_ids = set(annotation_promoter[annotation_promoter['Feature'] == rna]["transcript_id"].dropna())
            row['promoter_transcript_set'].update(transcript_ids)
    return row

df_all["Chromosome"]  = df_all["chrom"].str.replace("chr", "", regex=False)
df_all["PositionVCF"] = df_all["pos"]

df_all[annotation_columns] = 0
df_all['transcript_set']         = df_all.apply(lambda x: set(), axis=1)
df_all['promoter_transcript_set'] = df_all.apply(lambda x: set(), axis=1)

print(f"Running MANE annotation on {len(df_all):,} variants ...")
with Pool(4) as pool:
    results = list(tqdm(
        pool.imap(match_variants, [row for _, row in df_all.iterrows()]),
        total=len(df_all)
    ))
df_all = pd.DataFrame(results)
print("MANE annotation completed")
print(df_all[annotation_columns].sum())


## 6. Assign manuscript variant-type subgroups


In [ ]:
# Map annotation columns to manuscript subgroups.
SO_MISSENSE        = {"SO:0001583"}
SO_STOP_GAIN       = {"SO:0001587"}
SO_SYNONYMOUS      = {"SO:0001819", "SO:0001825"}
SO_SPLICE_DONOR    = {"SO:0001575"}
SO_SPLICE_ACCEPTOR = {"SO:0001574"}
SO_SPLICE_REGION   = {"SO:0001630", "SO:0001539"}

def assign_subgroup(row):
    so = str(row.get("mut_type_raw", "")).strip()
    if so in SO_MISSENSE:                          return "missense"
    if so in SO_STOP_GAIN:                         return "stop_gain"
    if so in SO_SYNONYMOUS:                        return "synonymous"
    if so in SO_SPLICE_DONOR | SO_SPLICE_ACCEPTOR: return "canonical_splice"
    if so in SO_SPLICE_REGION:                     return "canonical_splice"

    # Fallback: use MANE annotation columns.
    if row.get("coding_sequence", 0):
        if row.get("stop_codon", 0): return "stop_gain"
        return "missense"
    if row.get("mRNA_splice", 0):    return "canonical_splice"
    if row.get("mRNA_exon", 0):      return "synonymous"
    for rna in ["lncRNA", "snRNA", "antisenseRNA",
                "telomeraseRNA", "RNaseMRPRNA", "snoRNA"]:
        if row.get(rna, 0): return "rna_gene"
    return "other"

df_all["subgroup"] = df_all.apply(assign_subgroup, axis=1)

print("\nSubgroup by label:")
print(df_all.groupby(["subgroup", "label"]).size().unstack(fill_value=0))


## 7. Save and validate the benchmark


In [ ]:
CORE_COLUMNS = [
    "chrom", "pos", "ref", "alt", "gene", "label", "label_source",
    "subgroup", "cmc_tier", "af_bin", "hgvsc", "hgvsp", "transcript",
    "gnomad_af", "mut_type_raw",
]
output_columns = [column for column in CORE_COLUMNS + annotation_columns if column in df_all.columns]
benchmark = df_all[output_columns].copy()
benchmark["chrom"] = benchmark["chrom"].str.removeprefix("chr")
benchmark = benchmark.rename(columns={"chrom": "#CHROM", "pos": "POS", "ref": "REF", "alt": "ALT"})
benchmark = benchmark.sort_values(["#CHROM", "POS", "REF", "ALT"]).reset_index(drop=True)
benchmark.to_csv(OUT_PATH, index=False)

summary = (
    benchmark.groupby(["subgroup", "label"]).size().unstack(fill_value=0)
    .rename(columns={0: "benign", 1: "pathogenic"})
)
summary["total"] = summary.sum(axis=1)
summary.to_csv(SUMMARY_PATH)

if benchmark.duplicated(["#CHROM", "POS", "REF", "ALT"]).any():
    raise RuntimeError("Duplicate variant keys remain in the saved benchmark")
if set(benchmark["label"].unique()) != {0, 1}:
    raise RuntimeError("Both benchmark labels were not retained")

print(f"Saved {len(benchmark):,} variants to {OUT_PATH}")
print(f"Saved subgroup counts to {SUMMARY_PATH}")
display(summary)
